In [ ]:
import numpy as np
import tqdm
import matplotlib.pyplot as plt
from scipy.interpolate import RegularGridInterpolator
import sys
sys.path.append('../../theory/sedov_theory/python/')
from sedov_theory import SedovTalorProblem
import yt
from analysis_tool import CastroSimulation

In [ ]:
run_dir = '../run/'
file_start = 'plt_1d_'

In [ ]:
cs = CastroSimulation(run_dir, file_start)

# Check conservation of energy

In [ ]:
def compute_energy(time):
    """
    Compute energy in ergs/m

    Args:
        time: time in seconds

    Returns:
        energy in ergs/m
    """
    # Thermal and kinetic energy
    r, q, t = cs.extract_data( time, 'rho_E', 0)
    dr = r[1] - r[0]
    E_th_and_kin = 2*np.pi*dr * (q*r).sum() # erg/m

    # Ionization energy
    r, q, t = cs.extract_data( time, 'rho_H1', 0)
    M_ions = 2*np.pi*dr * (q*r).sum()
    E_ioniz = 13.6 * (q_e*1e7) * M_ions / (m_p * 1e3)

    return t, E_th_and_kin, E_ioniz

In [ ]:
t = []
E = []
E_ioniz = []
Etot = []
for time in tqdm.tqdm(np.linspace(0, 5e-9, 50)):
    t0, e, e_ioniz = compute_energy(time)
    t.append(t0)
    E.append(e)
    E_ioniz.append(e_ioniz)
    Etot.append(e + e_ioniz)
plt.plot(t, E)
plt.plot(t, E_ioniz, 'r-')
plt.plot(t, Etot, 'k--')
plt.grid()

# Compare data with theory

In [ ]:
# Calculate analytical solution
gamma = 5./3
rho = 1.67e-6 # g / cm^3
E = 4400 # erg / cm

sol = SedovTalorProblem(gamma, E, rho)

In [ ]:
plt.figure(figsize=(5, 4))
plt.clf()
level = 3
time = 4e-9

labels = {
    'rho_H0': 'Density of Hydrogen atoms',
    'rho_H1': 'Density of Hydrogen ions',
    'density': 'Total Hydrogen density',
}
colors = {
    'rho_H0': 'gray',
    'rho_H1': 'blue',
    'density': 'black'
}

for quantity in ['density', 'rho_H0', 'rho_H1']:
    r, q, t = cs.extract_data( time, quantity, level)
    # Convert r from cm to microns
    # Convert massic density in cc to number density in cc
    plt.plot(1e4*r, q/1.67e-6, '-o', label=labels[quantity], color=colors[quantity])

# Plot Sedov-Taylor solution
r_th = r
plt.plot( 1e4*r_th, sol.evaluate( 'density', r, t )/1.67e-6, 'k--', label='Sedov-Taylor theory' )
plt.legend(loc=1)

plt.ylabel('Density (10$^{18}$ cm$^{-3}$)')
plt.xlabel('r ($\mu m$)')
plt.title(f'Density at t={t*1.e9:.1f} ns')

In [ ]:
# plot data at difference resolutions
quantity = 'pressure'

plt.clf()
for level in range(3,-1,-1):
    r, q, t = cs.extract_data( 1.e-9, quantity, level)
    plt.plot(r, q, '-o', label='level=%d'%level)
    if level == 3:
        r_th = r
        t_th = t
#plt.plot( r_th, sol.evaluate( quantity, r_th, t_th ), 'k--', label='analytical' )
plt.legend(loc=0)

In [ ]:
# plot data at difference resolutions
quantity = 'Temp'

plt.clf()
for level in range(3,-1,-1):
    r, q, t = cs.extract_data( 80, quantity, level)
    plt.plot(r, q, '-o', label='level=%d'%level)
    if level == 3:
        r_th = r
        t_th = t
#plt.plot( r_th, sol.evaluate( quantity, r_th, t_th ), 'k--', label='analytical' )
plt.legend(loc=0)

In [ ]:
# Extract data from different time
# Note that time is not regularly spaced
q_arr = []
rmax_arr = []
for time in tqdm.tqdm( cs.output_times ):
    r, q, t = cs.extract_data(time, 'density', level=2)
    rmax = r[np.argmax(q)]
    rmax_arr.append(rmax)
    q_arr.append(q)
q_arr = np.stack(q_arr)
t_arr = cs.output_times.copy()
r_arr = r

# Interpolate on a grid with regularly-spaced time
interp = RegularGridInterpolator(points=(t_arr, r_arr), values=q_arr, bounds_error=False, fill_value=None)
t_interp, r_interp = np.meshgrid(
    np.linspace(0, t_arr.max(), 1000),
    np.linspace(0, r_arr.max(), 1000), indexing='ij')
q_interp = interp((t_interp, r_interp))

In [ ]:
plt.figure(figsize=(6, 2))
plt.imshow(q_interp.T/1.67e-6, origin='lower',
           extent=[0, t_arr.max()*1.e9, 0, r_arr.max()*1.e4],
           aspect='auto', vmax=3)
cb = plt.colorbar()
cb.set_label('H density (10$^{18}$ cm$^{-3}$)')
#plt.plot(t_arr, rmax_arr, 'r-')
r_analytical = sol.blast_radius(t_arr)
#plt.plot(t_arr*1.e9, r_analytical*1.e4, 'r--', label='Sedov-Taylor theory')
#plt.legend(loc=0)
plt.ylabel('r ($\mu m$)')
plt.xlabel('t (ns)')
plt.ylim(0,300)

In [ ]:
paths = [
#    'Castro_benchmark/2T_old_diffusion',
    'Castro_benchmark/2T_new_diffusion',
    'Comsol_benchmark/Broks_FixNuEI'
]

ls = [':', '-', '--']

labels = ['Castro', 'Comsol']

n_h = [ np.load(path + '_heavies.npy') for path in paths ]
n_e = [ np.load(path + '_electrons.npy') for path in paths ]
T_h = [ np.load(path + '_T_heavies.npy') for path in paths ]
T_e = [ np.load(path + '_T_electrons.npy') for path in paths ]

In [ ]:
def plot_density(it=500):
    r = np.linspace(0, 0.03*1.e4, 1001)
    for i in range(len(paths)):
        plt.plot( r, n_h[i][:,it], ls[i], label=labels[i] + ': heavies', lw=1, color='k')
        plt.plot( r, n_e[i][:,it], ls[i], label=labels[i] + ': electrons', lw=1, color='r')
    plt.ylim( 0, 4e24 )
    #plt.xlim(0,100)
    plt.legend(loc=0)
    plt.grid()
    plt.ylabel('Density (m^-3)')
    plt.xlabel('r (microns)')

In [ ]:
def plot_temperature(it=500):
    r = np.linspace(0, 0.03*1.e4, 1001)
    for i in range(len(paths)):
        plt.plot( r, T_h[i][:,it], ls[i], label=labels[i] + ': heavies', lw=1, color='k')
        plt.plot( r, T_e[i][:,it], ls[i], label=labels[i] + ': electrons', lw=1, color='r')
#    plt.ylim( 0, 4e24 )
    #plt.xlim(0,100)
    plt.legend(loc=0)
    plt.grid()
    plt.ylabel('Temperature (eV)')
    plt.xlabel('r (microns)')

In [ ]:
it = 100
plt.figure()
plt.clf()
plot_temperature(it)
plt.title('t = %.1f ns' %(10*it/1000) )
plt.grid()

In [ ]:
for it in tqdm.tqdm(range(0,1000,20)):
    plt.clf()
    plt.subplot(211)
    plot_density(it)
    plt.title('t = %.1f ns' %(10*it/1000) )
    plt.subplot(212)
    plot_temperature(it)
    plt.savefig('img/img%05d.png' %it)

In [ ]:
r = np.linspace(0, 0.03e-2, 1001)
dr = r[1] - r[0]
t = np.linspace(0,10, 1001)
plt.figure()
for i in range(len(labels)):
    plt.plot( t, (2*np.pi*r[:, np.newaxis]*dr*n_e[i]).sum(axis=0), label=labels[i], ls=ls[i], color='k')
plt.legend()
plt.ylabel('Number of electrons')
plt.xlabel('Time (ns)')
plt.grid()

In [ ]:
for iteration in range(0, 200, 2):
    print(iteration)
    ds = yt.load('../run/plt_2d_%05d/' %iteration, hint="castro")
    quantity = 'density'
    sl = yt.SlicePlot(ds, 'z', quantity)
    sl.set_log(quantity, False)
    sl.set_zlim(quantity, 0, 3e-6)
    sl.annotate_grids()
    sl.save('./img%05d.png' %iteration)

In [ ]:
ds = yt.load('../run/plt_2d_00400/', hint="castro")
quantity = 'rho_H1'
sl = yt.SlicePlot(ds, 'theta', quantity)
sl.set_log(quantity, False)
sl.annotate_grids()